# TikHarm: embedding extraction (frozen encoders)

Reads `manifest_full.csv` / `manifest_matched.csv` from the validation notebook and saves, for every clip:

| Feature | Encoder | Shape per clip |
|---|---|---|
| `vit` | ViT-B/16 (CLS token of 8 frames, evenly spaced in the first 10 s) | 8 x 768 |
| `w2v` | wav2vec2-base (5 time segments of the first 10 s) | 5 x 768 |
| `ast` | AST (5 time segments, pooled from patch tokens) | 5 x 768 |
| `ast_probs` | AST AudioSet class probabilities (music, speech, impacts, ...) | 527 |

Plus availability flags: `aud_ok` (audio >= 1 s), `n_frames_ok`, `decode_failed`.

**Order:** the 1,504 matched clips go first, then the rest of the full clean set. Chunks of 128 clips are
saved as they finish, and finished chunks are skipped, so you can stop and resume.

**Sensitive content:** this notebook never displays frames or audio. It only computes numbers.

**Settings:** GPU on, Internet on (to download the encoders).

In [ ]:
# Cell 1: config and manifests
import os, sys, json, time, subprocess, shutil, math
import numpy as np, pandas as pd
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

PREP = os.environ.get("PREP_DIR", "/kaggle/input/datasets/sanawerb/preparation")       # folder with the two manifests
WORK = os.environ.get("WORK_DIR", "/kaggle/working")
EMB = f"{WORK}/emb"
os.makedirs(EMB, exist_ok=True)

CFG = dict(win_s=10.0, n_frames=8, n_seg=5, sr=16000, chunk=128, n_threads=8, img=224)
SR, WIN, NF, K = CFG["sr"], CFG["win_s"], CFG["n_frames"], CFG["n_seg"]

full = pd.read_csv(f"{PREP}/manifest_full (1).csv")
matched = pd.read_csv(f"{PREP}/manifest_matched (1).csv")
assert full.uid.is_unique, "uid collision in manifest_full.csv"
assert set(matched.uid) <= set(full.uid), "matched set must be a subset of the full set"

# matched clips first, then the rest
order = pd.concat([full[full.uid.isin(matched.uid)], full[~full.uid.isin(matched.uid)]]).reset_index(drop=True)
order["in_matched"] = order.uid.isin(matched.uid).astype(int)
order[["uid", "split", "cls", "label", "in_matched"]].to_csv(f"{EMB}/order.csv", index=False)
print(len(order), "clips |", int(order.in_matched.sum()), "in the matched set")
print("paths exist:", order.path.map(os.path.exists).mean())

def pmap(fn, items):
    with ThreadPoolExecutor(CFG["n_threads"]) as ex:
        return list(ex.map(fn, items))

In [ ]:
print(order["path"].head(10).to_string(index=False))

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)
    if len(files) > 0:
        print("  files:", files[:5])

In [ ]:
for root, dirs, files in os.walk("/kaggle/input"):
    if files:
        print(root, "->", len(files), "files")

In [ ]:
print(order["path"].head(10).to_string(index=False))

In [ ]:
p = order["path"].iloc[0]

print("EXACT PATH:")
print(repr(p))

print("\nEXISTS:")
print(os.path.exists(p))

print("\nPARENT:")
print(os.path.dirname(p))

print("\nPARENT EXISTS:")
print(os.path.exists(os.path.dirname(p)))

In [ ]:
import os

print(os.listdir("/kaggle/input"))

In [ ]:
for root, dirs, files in os.walk("/kaggle/input"):
    if files:
        print(root, "->", len(files), "files")

In [5]:
import os

print("=== /kaggle/input ===")
print(os.listdir("/kaggle/input"))

print("\n=== Everything under /kaggle/input (first 100 paths) ===")
count = 0

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)

    count += 1
    if count >= 100:
        break

=== /kaggle/input ===
['datasets']

=== Everything under /kaggle/input (first 100 paths) ===
/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/sanawerb
/kaggle/input/datasets/sanawerb/preparation


In [3]:
import os

matches = []

for root, dirs, files in os.walk("/kaggle/input"):
    if os.path.basename(root).lower() == "dataset":
        matches.append(root)

print("Dataset folders found:")
for x in matches:
    print(x)

Dataset folders found:


In [4]:
DATASET_ROOT = "/kaggle/input/tikharm-dataset/Dataset"

print(os.listdir(DATASET_ROOT))

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/tikharm-dataset/Dataset'